### Import the libraries

In [461]:
import spacy
from spacy import tokenizer
from bs4 import BeautifulSoup
import nltk
import string
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize.toktok import ToktokTokenizer
import pandas as pd

### Load the covid19 tweet data

In [462]:
# Read the data from the Excel and load into the dataframe using pandas
rawData =pd.read_excel("COVID_19_vaccine_100.xlsx")
rawData.columns=['tweets']
rawData.head(5)

,tweets
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...
1,May i remind you that the vaccine isnt just fo...
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik..."
3,"Shandro, Hinshaw to give COVID-19 vaccine upda..."
4,You ever Noticed This? They just announced Yes...


### NULL Check

In [463]:
# Check nulls and if it hampers processing drop them
rawData['tweets'].isnull().sum()

0

## Text Processing

#### Removing html tags

<span style="color:white">Often, unstructured text contains a lot of noise, especially if you use techniques like web or screen scraping. HTML tags are typically one of these components which don’t add much value towards understanding and analyzing text.</span>

In [464]:
def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text
strip_html_tags('<html><h2>May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO</h2></html>')

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Removing accented characters
<span style="color:white">Usually in any text corpus, you might be dealing with accented characters/letters, especially if you only want to analyze the English language. Hence, we need to make sure that these characters are converted and standardized into ASCII characters. A simple example — converting é to e.</span>

In [465]:
import unicodedata


def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text

remove_accented_chars('Sómě Áccěntěd těxt')
remove_accented_chars("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

### Contraction MAP

In [466]:
#Contraction Mapping
CONTRACTION_MAP = {
"ain't": "is not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he'll've": "he he will have",
"he's": "he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how is",
"I'd": "I would",
"I'd've": "I would have",
"I'll": "I will",
"I'll've": "I will have",
"I'm": "I am",
"I've": "I have",
"i'd": "i would",
"i'd've": "i would have",
"i'll": "i will",
"i'll've": "i will have",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'd've": "it would have",
"it'll": "it will",
"it'll've": "it will have",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she would",
"she'd've": "she would have",
"she'll": "she will",
"she'll've": "she will have",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so as",
"that'd": "that would",
"that'd've": "that would have",
"that's": "that is",
"there'd": "there would",
"there'd've": "there would have",
"there's": "there is",
"they'd": "they would",
"they'd've": "they would have",
"they'll": "they will",
"they'll've": "they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what'll've": "what will have",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"when's": "when is",
"when've": "when have",
"where'd": "where did",
"where's": "where is",
"where've": "where have",
"who'll": "who will",
"who'll've": "who will have",
"who's": "who is",
"who've": "who have",
"why's": "why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you would",
"you'd've": "you would have",
"you'll": "you will",
"you'll've": "you will have",
"you're": "you are",
"you've": "you have"
}

#### Expanding Contractions
<span style="color:white">Contractions are shortened version of words or syllables. They often exist in either written or spoken forms in the English language. These shortened versions or contractions of words are created by removing specific letters and sounds. In case of English contractions, they are often created by removing one of the vowels from the word. Examples would be, do not to don’t and I would to I’d. Converting each contraction to its expanded, original form helps with text standardization.</span>

In [467]:
import re
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP):

    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())),
                                      flags=re.IGNORECASE|re.DOTALL)
    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = contraction_mapping.get(match)\
                                if contraction_mapping.get(match)\
                                else contraction_mapping.get(match.lower())
        expanded_contraction = first_char+expanded_contraction[1:]
        return expanded_contraction

    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'", "", expanded_text)
    return expanded_text

expand_contractions("Y'all are enjoying this class we'd think")

'You all are enjoying this class we would think'

#### Removing Special Characters
<span style="color:white">Special characters and symbols are usually non-alphanumeric characters or even occasionally numeric characters (depending on the problem), which add to the extra noise in unstructured text. Usually, simple regular expressions (regexes) can be used to remove them.</span>

In [468]:
def remove_special_characters(text, remove_digits=False):
    pattern = r'[^a-zA-z0-9\s]' if not remove_digits else r'[^a-zA-z\s]'
    text = re.sub(pattern, ' ', text)
    return text

remove_special_characters("Well this was fun! What do you think? 123#@! covid-19 covid_19", remove_digits=False)

'Well this was fun  What do you think  123    covid 19 covid_19'

#### Removing Stopwords
<span style="color:white">Words which have little or no significance, especially when constructing meaningful features from text, are known as stopwords or stop words. These are usually words that end up having the maximum frequency if you do a simple term or word frequency in a corpus. Typically, these can be articles, conjunctions, prepositions and so on. Some examples of stopwords are a, an, the, and the like.</span>

In [469]:
# Using nltk , we want to remove the stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
#Let us print the stopwords and observe what they are
set(stopwords.words('english'))
stopword_list= stopwords.words('english')

from nltk.tokenize.toktok import ToktokTokenizer
tokenizer = ToktokTokenizer()

def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

remove_stopwords("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'May remind vaccine isnt caught covid-19 , also prevent ................ https://t.co/htFc4jP4CO'

#### Remomving URLS
<span style="color:white">Words which contains urls are mostly not required for the data analysis</span>

In [470]:
# def url_removal(text):
#     modified_text = ' '.join([words for words in text.split(' ') if words[0:4]!="http"])
#     return modified_text
#
# url_removal("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO" )


### Lemmatization

In [471]:
def lemmatize_text(text):
    text = nlp(text)
    text = ' '.join([word.lemma_ if word.lemma_ != '-PRON-' else word.text for word in text])
    return text
lemmatize_text("May i remind you that the vaccine isnt just for those who caught covid-19")

'may I remind you that the vaccine be not just for those who catch covid-19'

### Stemming

In [472]:
ps = nltk.PorterStemmer()
def simple_stemmer(text):
    text = ' '.join([ps.stem(word) for word in text.split()])
    return text
simple_stemmer("May i remind you that the vaccine isnt just for those who caught covid-19")

'may i remind you that the vaccin isnt just for those who caught covid-19'

## Normalization

In [473]:
#Wrapping all the above processes
def normalize_corpus(doc, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True,
                     text_lemmatization=True, special_char_removal=True,
                     stopword_removal=True, remove_digits=False):
    # normalize each document in the corpus
    # strip HTML
    if html_stripping:
        doc = strip_html_tags(doc)
    # remove accented characters
    if accented_char_removal:
        doc = remove_accented_chars(doc)
    # expand contractions
    if contraction_expansion:
        doc = expand_contractions(doc)
    # lowercase the text
    if text_lower_case:
        doc = doc.lower()
    # remove extra newlines
    doc = re.sub(r'[\r|\n|\r\n]+', ' ',doc)
    # lemmatize text
    if text_lemmatization:
        doc = lemmatize_text(doc)
    # remove special characters and\or digits
    if special_char_removal:
        # insert spaces between special characters to isolate them
        special_char_pattern = re.compile(r'([{.(-)!}])')
        doc = special_char_pattern.sub(" \\1 ", doc)
        doc = remove_special_characters(doc, remove_digits=remove_digits)
    # remove extra whitespace
    doc = re.sub(' +', ' ', doc)
    # remove stopwords
    if stopword_removal:
        doc = remove_stopwords(doc, is_lower_case=text_lower_case)
    # ' '.join(doc)

    return doc

rawData['tweets_cleaned']=rawData['tweets'].apply(lambda x:normalize_corpus(x))


C:\Users\ragha\AppData\Local\Temp\ipykernel_9112\172976477.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


In [474]:
rawData


,tweets,tweets_cleaned
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...,johensley darcyshepherd13 jeffreyguterman real...
1,May i remind you that the vaccine isnt just fo...,may I remind vaccine catch covid 19 also preve...
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik...",mjackermanmdphd hi dr ackerman I like know go ...
3,"Shandro, Hinshaw to give COVID-19 vaccine upda...",shandro hinshaw give covid 19 vaccine update n...
4,You ever Noticed This? They just announced Yes...,ever notice announce yesterday december 13th 2...
...,...,...
95,Must viewing: Principles of vaccines programs ...,must view principle vaccine program control co...
96,COVID-19 vaccine's protection against virus ou...,covid 19 vaccine protection virus outweighs po...
97,I've spent some time today looking into whethe...,I spend time today look whether covid 19 vacci...
98,COVID-19 Vaccine Likely Beneficial For Breastf...,covid 19 vaccine likely beneficial breastfed b...


### POS Tagging

In [475]:
import spacy
nltk.download('averaged_perceptron_tagger')
# Download NLTK Punkt sentence tokenizer
import nltk

# nltk.download('all')
nlp = spacy.load("en_core_web_sm")
# pos_tagged_data=[]
# for sentence in rawData['tweets_cleaned']:
#     # print(sentence)
#     sentence_nlp = nlp(sentence)
#     # print(sentence_nlp,end='\n')
#     spacy_pos_tagged=[]
#     spacy_pos_tagged_sentence = [(word.__str__().strip(), word.tag_, word.pos_,sentence) for word in [i for i in sentence_nlp]]
#     pos_tagged_data=pos_tagged_data+spacy_pos_tagged_sentence
#
#
# spacy_pos_tagged_data = pd.DataFrame(pos_tagged_data,columns=['word','pos_tag','tag_type','sentence'])
# spacy_pos_tagged_data.to_csv('spacy_pos_tagged_data.csv')
# spacy_pos_tagged_data.head(20)
nltk.download('averaged_perceptron_tagger')
# Download NLTK Punkt sentence tokenizer
nltk.download('punkt')

# demo for POS tagging for sample news headline
spacy_pos_tagged = []
for sentence in rawData['tweets_cleaned']:
    sentence_nlp = nlp(sentence)
    print(sentence_nlp)
    # POS tagging with Spacy
    spacy_pos_tagged_temp = [(word.__str__().strip(), word.tag_, word.pos_) for word in sentence_nlp]
    spacy_pos_tagged = spacy_pos_tagged+ spacy_pos_tagged_temp
spacy_pos_tagged_data = pd.DataFrame(spacy_pos_tagged, columns=['Word', 'POS tag', 'Tag type'])


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


johensley darcyshepherd13 jeffreyguterman realdonaldtrump different vaccine different ingredient I take covid 19 second 95 efficacy fine I also thevaccine make recombinant dna make mrna I really hate people spread misinformation luciferase enzyme vaccine
may I remind vaccine catch covid 19 also prevent https co htfc4jp4co
mjackermanmdphd hi dr ackerman I like know go discuss safe covid 19 vaccine lqts patient
shandro hinshaw give covid 19 vaccine update noon https co 8jhtbfjqku
ever notice announce yesterday december 13th 2020 great vaccine protect covid 19 95 effective take less year develop yet still cure cancer year
covid_long longcovid anaphylaxis troublesome 20 year problem ade cv vaccine covid 19 vaccine design elicit neutralizing antibody may sensitize vaccine recipient severe disease vaccinate https co gnjlljalju
b911 poll plan receive type covid 19 vaccine covidvaccine
covid 19 vaccine safety efficacy important brand adoctor https co 5bewszrc3e
12 hour post 2nd covid vaccine b

## Question 1:
<span color:white>Determine the part-of-speech (POS) tag of the word 'covid' in an entire dataset after text processing. Additionally, could you calculate the total number of times the word 'covid' appears in the text processed column and provide its frequency for each POS tag in which it occurs?" For example, it may occur as a proper noun (NNP), verb (VB), cardinal digit (CD), etc. Can you display a tweet from each category?</span>

### Q1 Frequency of the word covid appears in the dataset:

## Count of words containing 'covid' as a substring or as a word it self:

In [476]:
print("Frequency of the word 'covid' is: {}".format(spacy_pos_tagged_data[spacy_pos_tagged_data['Word'].str.contains('covid')]['Word'].count()))
spacy_pos_tagged_data[spacy_pos_tagged_data['Word'].str.contains('covid')]

Frequency of the word 'covid' is: 118


,Word,POS tag,Tag type
10,covid,JJ,ADJ
38,covid,NN,NOUN
55,covid,JJ,ADJ
63,covid,JJ,ADJ
81,covid,JJ,ADJ
...,...,...,...
1988,covid,NN,NOUN
1995,covid,JJ,ADJ
2016,covid,JJ,ADJ
2036,covid,JJ,ADJ


The Frequency of the word 'covid' contain as a substring of a word or as  a word itself is: 118.
The above code resulting the covid word frequency with its respective POS tag

## Count of words matches exactly with the word 'covid':

In [477]:
print("Frequency of the word 'covid' is: {}".format(spacy_pos_tagged_data[spacy_pos_tagged_data['Word']=='covid']['Word'].count()))
spacy_pos_tagged_data[spacy_pos_tagged_data['Word']=='covid']

Frequency of the word 'covid' is: 105


,Word,POS tag,Tag type
10,covid,JJ,ADJ
38,covid,NN,NOUN
55,covid,JJ,ADJ
63,covid,JJ,ADJ
81,covid,JJ,ADJ
...,...,...,...
1988,covid,NN,NOUN
1995,covid,JJ,ADJ
2016,covid,JJ,ADJ
2036,covid,JJ,ADJ


The Frequency of the word 'covid' in column 'Word' is: 105.
The above code resulting the covid word frequency with its respective POS tag

### Q1B Word count of 'covid' across each pos_tag type

In [478]:
covid_word_pos_tags = spacy_pos_tagged_data[spacy_pos_tagged_data['Word']=='covid'].groupby(
    ['Word','POS tag']).size().reset_index(name='counts').sort_values('counts',ascending=False)
covid_word_pos_tags

,Word,POS tag,counts
0,covid,JJ,61
1,covid,NN,21
2,covid,NNP,15
3,covid,VB,6
4,covid,VBP,2


The result yielded is a grouped data on the column Word where word is equal to 'covid'.
We can observe the POS tag type JJ has the most frequent count

### Question 2:
<span color:white>What is the most frequently occurring entity name associated with the "CARDINAL" entity type? Could you also specify the frequency of that entity name in the dataset after text preprocessing?</span>

## Cardinal entity(CD) Count
### Fetch the CD type entity match from POS tag

In [544]:
spacy_pos_tagged_data[spacy_pos_tagged_data['POS tag']=='CD']

,Word,POS tag,Tag type
11,19,CD,NUM
13,95,CD,NUM
39,19,CD,NUM
56,19,CD,NUM
64,19,CD,NUM
...,...,...,...
1989,19,CD,NUM
1996,19,CD,NUM
2017,19,CD,NUM
2037,19,CD,NUM


Here we can observe that CARDINAL values includes digits as well as numbers in words.
Note:
The remove_special_charaters function is lightly modified to replace the special characters with space instead of ''
    sub('_','',covid_19) gives covid19
    sub('_',' ',covid_19) gives covid 19
 THis gives us the exact math to the word covid properly.

### Count of each word with respect to 'CD' POS tag type in descending order

In [480]:
spacy_pos_tagged_data[spacy_pos_tagged_data['POS tag']=='CD'].groupby(["Word","POS tag"]).size().reset_index(name='counts').sort_values('counts', ascending=False)

,Word,POS tag,counts
12,19,CD,102
15,2,CD,7
59,one,CD,6
5,11,CD,5
23,3,CD,4
...,...,...,...
30,41,CD,1
1,001,CD,1
34,4nuedel1cj,CD,1
36,54,CD,1


From the above we can observe the Number 19 is repeated most frequently among the others in the data set.

### Question 3
<span color:white>Identify the top 5 most commonly appearing entity types in the provided dataset and determine
their respective frequencies using the Spacy Named Entity Recognition (NER).</span>

### NER

### Five most commonly appearing entity types

In [481]:
from spacy import displacy
sentence = str(rawData.iloc[2].tweets_cleaned)
sentence_nlp = nlp(sentence)

# print named entities in article
print([(word, word.ent_type_) for word in sentence_nlp if word.ent_type_])

# visualize named entities
displacy.render(sentence_nlp, style='ent', jupyter=True)
#
#
# ner_tags=[]
# for i in rawData['tweets_cleaned']:
#     doc = nlp(i)
#     if doc.ents:
#         ner_tags= ner_tags+[(ent.text,ent.label_,word.tag_) for word,ent in zip(doc,doc.ents)]
#     # else:
#     #     print(i)
#
# ner_tags_data =pd.DataFrame(ner_tags,columns=['text','label','pos_tag'])
# ner_tags_data

[(dr, 'PERSON'), (ackerman, 'PERSON'), (19, 'CARDINAL')]


In [482]:
named_entities = []
for sentence in rawData['tweets_cleaned']:
    temp_entity_name = ''
    temp_named_entity = None
    sentence = nlp(sentence)
    for word in sentence:
        term = word.text
        tag = word.ent_type_
        if tag:
            temp_entity_name = ' '.join([temp_entity_name, term]).strip()
            temp_named_entity = (temp_entity_name, tag)
        else:
            if temp_named_entity:
                named_entities.append(temp_named_entity)
                temp_entity_name = ''
                temp_named_entity = None

entity_frame = pd.DataFrame(named_entities,
                            columns=['Entity Name', 'Entity Type'])
entity_frame.groupby(['Entity Type']).size().reset_index(name='count').sort_values('count',ascending=False).head(20)

,Entity Type,count
0,CARDINAL,130
8,PERSON,67
1,DATE,17
7,ORG,12
2,GPE,10
5,NORP,6
6,ORDINAL,3
10,QUANTITY,3
9,PRODUCT,2
11,TIME,2


From the above we can observe the Entity type CARDINAL is occurred most frequently in the entire dataset. Latter the word PERSON.

### Question 4
<span color:white>Perform Sentiment Analysis and write python code to provide a total count of positive and negative sentiments.</span>

### Sentiment Analysis

In [483]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
#Demo check on first row of the data
analyzer.polarity_scores(rawData['tweets_cleaned'][0])

{'neg': 0.179, 'neu': 0.769, 'pos': 0.051, 'compound': -0.6697}

In [484]:
# function to return either 1 0r 0 for the pos score of the text
def get_sentiment(text):

    scores = analyzer.polarity_scores(text)

    sentiment = 1 if scores['pos'] > 0 else 0

    return sentiment

### Adding new sentiment column to store either positive or zero value of the scores
rawData['sentiment'] = rawData['tweets_cleaned'].apply(lambda x :get_sentiment(x))


print('Sentiment Analysis: \n','Positive Tweets: {}, Negative Tweets: {}'.format(rawData[rawData['sentiment']==1]['sentiment'].count(),rawData[rawData['sentiment']==0]['sentiment'].count()))

Sentiment Analysis: 
 Positive Tweets: 52, Negative Tweets: 48


### Question 5
Compute the topic modeling LDA on the tweets. What is the main sentiment of the 1st topic identified by LDA? Can you display a sample tweet with the topic?

### Topic Modelling

In [545]:
from pyLDAvis import gensim
#! pip install pyLDAvis
# Importing modules
import pandas as pd
import re
import numpy as np
import pandas as pd
from pprint import pprint

# Gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

# spacy for lemmatization
import spacy

# Plotting tools
import pyLDAvis
import pyLDAvis.gensim  # don't skip this
import matplotlib.pyplot as plt
%matplotlib inline


import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

def sent_to_words(sentence):
    data = yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations
    return data

rawData['tweets_cleaned_words']=rawData['tweets_cleaned'].apply(lambda x:list(sent_to_words(x))[0])
rawData['tweets_cleaned_words']

0     [johensley, darcyshepherd, jeffreyguterman, re...
1     [may, remind, vaccine, catch, covid, also, pre...
2     [mjackermanmdphd, hi, dr, ackerman, like, know...
3     [shandro, hinshaw, give, covid, vaccine, updat...
4     [ever, notice, announce, yesterday, december, ...
                            ...                        
95    [must, view, principle, vaccine, program, cont...
96    [covid, vaccine, protection, virus, outweighs,...
97    [spend, time, today, look, whether, covid, vac...
98    [covid, vaccine, likely, beneficial, breastfed...
99    [alright, guy, want, make, trip, pact, cuz, ia...
Name: tweets_cleaned_words, Length: 100, dtype: object

The above result is the tokenization column of rawData['tweets'] contains in the form of list of words which will be further utilized for generating Bigrams and Trigrams

In [546]:
data_words = rawData['tweets_cleaned']
# Build the bigram and trigram models
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)

# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

# See bigram example
print(bigram_mod[bigram_mod[data_words]])

['johensley darcyshepherd13 jeffreyguterman realdonaldtrump different vaccine different ingredient I take covid 19 second 95 efficacy fine I also thevaccine make recombinant dna make mrna I really hate people spread misinformation luciferase enzyme vaccine', 'may I remind vaccine catch covid 19 also prevent https co htfc4jp4co', 'mjackermanmdphd hi dr ackerman I like know go discuss safe covid 19 vaccine lqts patient', 'shandro hinshaw give covid 19 vaccine update noon https co 8jhtbfjqku', 'ever notice announce yesterday december 13th 2020 great vaccine protect covid 19 95 effective take less year develop yet still cure cancer year', 'covid_long longcovid anaphylaxis troublesome 20 year problem ade cv vaccine covid 19 vaccine design elicit neutralizing antibody may sensitize vaccine recipient severe disease vaccinate https co gnjlljalju', 'b911 poll plan receive type covid 19 vaccine covidvaccine', 'covid 19 vaccine safety efficacy important brand adoctor https co 5bewszrc3e', '12 hou

In [547]:
# Define functions for stopwords, bigrams, trigrams and lemmatization

# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]

def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent))
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

The above function performs data pre-processing which removes stop words  and lemmatize the words.

### Topic Modelling using LDA

In [548]:
# Remove Stop Words
data_words_nostops = remove_stopwords(data_words)

# Form Bigrams
data_words_bigrams = make_bigrams(data_words_nostops)

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
# python3 -m spacy download en
# nlp = spacy.load('en_core_web_lg', disable=['parser', 'ner'])
nlp = spacy.load("en_core_web_sm")

# Do lemmatization keeping only noun, adj, vb, adv
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

print(data_lemmatized[0])


['realdonaldtrump', 'different', 'vaccine', 'different', 'ingredient', 'take', 'covid', 'second', 'efficacy', 'fine', 'also', 'thevaccine', 'make', 'recombinant', 'make', 'mrna', 'really', 'hate', 'people', 'spread', 'misinformation', 'luciferase', 'vaccine']


In [549]:
 #Create Dictionary
id2word = corpora.Dictionary(data_lemmatized)

# Create Corpus
texts = data_lemmatized

# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]


id2word function creates a tuplw with id and the word which is loaded into corpus.This form of data is used as a hyper parameter in LDA model to create the tuple of ids with respective words.

In [550]:
# Build LDA model
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=2,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha='auto',
                                           per_word_topics=True)

#chunksize controls how many documents are processed at a time in the training algorithm.
#Increasing chunksize will speed up training, at least as long as the chunk of documents easily fit into memory.
#passes controls how often we train the model on the entire corpus (set to 10).
#Another word for passes might be “epochs”.
#Iterations is somewhat technical, but essentially it controls how often we repeat a particular loop over each document.
#It is important to set the number of “passes” and “iterations” high enough.

In [551]:
# Print the Keyword in the topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

[(0,
  '0.056*"vaccine" + 0.034*"covid" + 0.011*"say" + 0.008*"question" + '
  '0.008*"work" + 0.008*"dose" + 0.007*"expert" + 0.007*"effect" + '
  '0.007*"efficacy" + 0.007*"virus"'),
 (1,
  '0.064*"vaccine" + 0.040*"covid" + 0.012*"safe" + 0.010*"people" + '
  '0.009*"get" + 0.007*"effect" + 0.007*"test" + 0.007*"co" + '
  '0.006*"effective" + 0.006*"take"')]


### Main sentiment of the 1st topic identified by LDA

<span color:white>AFINN is an English word listed developed by Finn Årup Nielsen. Words scores range from minus five (negative) to plus five (positive). The English language dictionary consists of 2,477 coded words.</span>

In [583]:
# Compute sentiment analysis for the main topic
!pip install afinn
from afinn import Afinn
afinn = Afinn(language='en')
topic_id = 0
topic_words = lda_model.show_topic(topic_id, topn=10)
count= 0
ll =[]
for word, _ in topic_words:
    ll.append(word)
    count =count+afinn.score(word.strip())

if count>0:
    print('Predicted Topic 1 main sentiment is : Positive')
else:
    print('Predicted Topic 1 main sentiment is : Negative')

print('Topic words: ',ll)

Predicted Topic 1 main sentiment is : Negative
Topic words:  ['vaccine', 'covid', 'say', 'question', 'work', 'dose', 'expert', 'effect', 'efficacy', 'virus']


In [579]:
# Compute Perplexity
print('\nPerplexity: ', lda_model.log_perplexity(corpus))  # a measure of how good the model is. lower the better.

# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Perplexity:  -6.228541850348731

Coherence Score:  0.4252763011265944


In [580]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

PreparedData(topic_coordinates=              x    y  topics  cluster       Freq
topic                                           
0      0.063245  0.0       1        1  57.027752
1     -0.063245  0.0       2        1  42.972248, topic_info=         Term      Freq     Total Category  logprob  loglift
26       safe  6.000000  6.000000  Default  30.0000  30.0000
116       say  7.000000  7.000000  Default  29.0000  29.0000
148  question  5.000000  5.000000  Default  28.0000  28.0000
156      dose  5.000000  5.000000  Default  27.0000  27.0000
104       get  6.000000  6.000000  Default  26.0000  26.0000
..        ...       ...       ...      ...      ...      ...
95     answer  2.180010  3.907057   Topic2  -5.4578   0.2612
17       take  2.824948  7.287364   Topic2  -5.1986  -0.1030
125     trial  2.182851  4.596501   Topic2  -5.4565   0.1000
76        new  2.202993  5.973854   Topic2  -5.4473  -0.1530
3    efficacy  2.194627  6.664488   Topic2  -5.4511  -0.2662

[130 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
154       1  0.982308  african
95        1  0.511894   answer
95        2  0.511894   answer
239       2  1.050352    apply
96        2  1.050476      ask
...     ...       ...      ...
80        1  0.818735     work
80        2  0.163747     work
379       2  1.050454   worker
42        1  0.743277     year
42        2  0.247759     year

[123 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2])